In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
train_dir="chest_xray/train"
test_dir="chest_xray/test"



train_datagen = ImageDataGenerator(
    rescale=1./255, 
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [3]:
train_func = train_datagen.flow_from_directory(
    train_dir, 
    target_size=(224, 224), 
    batch_size=32,
    class_mode='binary' 
)


test_func = test_datagen.flow_from_directory(
    test_dir, 
    target_size=(224, 224), 
    batch_size=32,
    class_mode='binary' 
)

Found 5232 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import ResNet50

In [5]:
resnet = ResNet50(weights='imagenet', input_shape=(224, 224, 3))

x = Flatten()(resnet.output) 
x = Dense(128, activation='relu')(x)  
x = Dropout(0.5)(x)  
x = Dense(1, activation='sigmoid')(x) 

resnet_model = Model(inputs=resnet.input, outputs=x)

resnet_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

2026-02-26 09:40:50.492559: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-02-26 09:40:50.492591: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-02-26 09:40:50.492602: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-02-26 09:40:50.492834: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-26 09:40:50.493169: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [9]:
history = resnet_model.fit(
    train_func, 
    epochs=5, 
    validation_data=test_func  
)

Epoch 1/5
164/164 [==============================] - 178s 1s/step - loss: 0.4079 - accuracy: 0.8127 - val_loss: 0.6481 - val_accuracy: 0.6250
Epoch 2/5
164/164 [==============================] - 217s 1s/step - loss: 0.2638 - accuracy: 0.8905 - val_loss: 1.5184 - val_accuracy: 0.6250
Epoch 3/5
164/164 [==============================] - 212s 1s/step - loss: 0.2360 - accuracy: 0.9012 - val_loss: 1.6099 - val_accuracy: 0.6250
Epoch 4/5
164/164 [==============================] - 227s 1s/step - loss: 0.2250 - accuracy: 0.9104 - val_loss: 1.7147 - val_accuracy: 0.6250
Epoch 5/5
164/164 [==============================] - 170s 1s/step - loss: 0.2246 - accuracy: 0.9100 - val_loss: 1.6108 - val_accuracy: 0.6250


In [10]:
resnet_model.save("ass2.h5")

/Users/ashishshirsath/tf-cnn-env/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
resnet_model.compile(optimizer='"sgd"', loss='hinge', metrics=['accuracy'])

In [ ]:
history = resnet_model.fit(
    train_func, 
    epochs=5, 
    validation_data=test_func  
)

Epoch 1/2
164/164 [==============================] - 213s 1s/step - loss: 0.5164 - accuracy: 0.7422 - val_loss: 0.7517 - val_accuracy: 0.6234
Epoch 2/2
164/164 [==============================] - 227s 1s/step - loss: 0.5162 - accuracy: 0.7422 - val_loss: 0.7503 - val_accuracy: 0.6250


In [ ]:
resnet_model.save("ass2-1.h5")

In [6]:
resnet_model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

In [7]:
history = resnet_model.fit(
    train_func, 
    epochs=5, 
    validation_data=test_func  
)

Epoch 1/5


2026-02-26 09:40:59.715809: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


164/164 [==============================] - 124s 735ms/step - loss: 0.6265 - accuracy: 0.7376 - val_loss: 0.6623 - val_accuracy: 0.6250
Epoch 2/5
164/164 [==============================] - 138s 839ms/step - loss: 0.5803 - accuracy: 0.7422 - val_loss: 0.6718 - val_accuracy: 0.6250
Epoch 3/5
164/164 [==============================] - 179s 1s/step - loss: 0.5660 - accuracy: 0.7422 - val_loss: 0.6776 - val_accuracy: 0.6250
Epoch 4/5
164/164 [==============================] - 171s 1s/step - loss: 0.5085 - accuracy: 0.7422 - val_loss: 0.6968 - val_accuracy: 0.6250
Epoch 5/5
164/164 [==============================] - 204s 1s/step - loss: 0.4314 - accuracy: 0.7441 - val_loss: 0.7381 - val_accuracy: 0.6250


In [9]:
from tensorflow.keras.models import load_model

model = load_model("ass2.h5", compile=False)

In [10]:
y_true = test_func.classes

In [11]:
test_func.reset()

In [12]:
y_pred_prob = model.predict(test_func)
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_true, y_pred_prob)
print("AUC:", auc)

20/20 [==============================] - 19s 868ms/step
AUC: 0.49106947183870264
